# Criando um transformer do zero

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import math
import numpy as np
import re

torch.manual_seed(23)

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [3]:
MAX_SEQ_LEN = 128 #30

In [4]:
class PossitionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len=MAX_SEQ_LEN):
        super().__init__()
        self.pos_embed_matrix = torch.zeros(max_seq_len, d_model, device=device)
        token_pos = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term  = torch.exp(torch.arange(0,d_model, 2).float()*(-math.log(10000.0)/d_model) )
        
        self.pos_embed_matrix[:, 0::2] = torch.sin(token_pos * div_term)
        self.pos_embed_matrix[:, 1::2] = torch.cos(token_pos * div_term)
        self.pos_embed_matrix = self.pos_embed_matrix.unsqueeze(0).transpose(0,1)

    def forward(self, x):
        #print(self.pos_embed_matrix.shape)
        #print(x.shape)
        return x + self.pos_embed_matrix[:x.size(0), :]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model = 512, num_heads=8):
        super().__init__()
        assert d_model % num_heads == 0, 'Embedding size not compatible with num_heads'
        
        self.d_v = d_model // num_heads
        self.d_k = self.d_v
        self.num_heads = num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)


    def forward(self, Q, K, V, mask = None):
        batch_size = Q.size(0)
        """
        Q, K, V -> [bacth size, seq_len, num_heads*d_k]
        after transpose Q, k; V -> [batch size, num_heads, seq_len, dk]
        """
        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)

        weighted_values, attention = self.scale_dot_product(Q, K, V, mask)
        weighted_values = weighted_values.transpose(1,2).contiguous().view(batch_size, -1, self.num_heads*self.d_k)
        weighted_values = self.W_o(weighted_values)

        return weighted_values, attention

    
    def scale_dot_product(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2,-1)) /math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attention = F.softmax(scores, dim=-1)
        weighted_values= torch.matmul(attention, V)

        return weighted_values, attention

class PositionFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))

class EncoderSubLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn       = PositionFeedForward(d_model, d_ff)
        self.norm1     = nn.LayerNorm(d_model)
        self.norm2     = nn.LayerNorm(d_model)
        self.dropout1  = nn.Dropout(dropout)
        self.dropout2  = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attention_score, _ = self.self_attn(x, x, x, mask)
        
        x = x + self.dropout1(attention_score)
        x = self.norm1(x)

        x = x + self.dropout2(self.ffn(x))
        return self.norm2(x)

class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([EncoderSubLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

class DecoderSubLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.droput1 = nn.Dropout(dropout)
        self.droput2 = nn.Dropout(dropout)
        self.droput3 = nn.Dropout(dropout)

    def forward(self, x, encoder_output, target_mask=None, encoder_mask=None):
        attention_score, _ = self.self_attn(x, x, x, target_mask)
        x = x + self.droput1(attention_score)
        x = self.norm1(x)

        encoder_attn, _ = self.cross_attn(x, encoder_output, encoder_output, encoder_mask)
        x = x + self.droput2(encoder_attn)
        x = self.norm2(x)

        ff_output = self.feed_forward(x)
        x = x + self.droput3(ff_output)
        return self.norm3(x)

class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([DecoderSubLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, encoder_output, target_mask, encoder_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, target_mask, encoder_mask)
        return self.norm(x)

In [5]:
class Transformer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers,
                input_vocab_size, target_vocab_size,
                max_len=MAX_SEQ_LEN, dropout=0.1):
        super().__init__()
        self.encoder_embedding = nn.Embedding(input_vocab_size,  d_model)
        self.decoder_embedding = nn.Embedding(target_vocab_size, d_model)
        self.pos_embedding     = PossitionalEmbedding(d_model, max_len)
        self.encoder           = Encoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder           = Decoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.output_layer       = nn.Linear(d_model, target_vocab_size)

    def forward(self, source, target):
        # masks do encoder 
        source_mask, target_mask = self.mask(source, target)
        # Embedding e encoding positional
        source = self.encoder_embedding(source) * math.sqrt(self.encoder_embedding.embedding_dim)
        source = self.pos_embedding(source)
        # Encoder
        encoder_output = self.encoder(source, source_mask)
        
        # Decoder embedding & positional encoding
        target = self.decoder_embedding(target) * math.sqrt(self.decoder_embedding.embedding_dim)
        target = self.pos_embedding(target)
        #Decoder
        output = self.decoder(target, encoder_output, target_mask, source_mask)

        return self.output_layer(output)


    def mask(self, source, target):
        source_mask = (source != 0 ).unsqueeze(1).unsqueeze(2)
        target_mask = (target != 0 ).unsqueeze(1).unsqueeze(2)

        size = target.size(1)
        no_mask = torch.tril(torch.ones((1, size, size), device=device)).bool()
        target_mask = target_mask & no_mask

        return source_mask, target_mask

# Simple test

In [6]:
seq_len_source = 10
seq_len_target = 10
batch_size = 2
input_vocab_size = 50
target_vocab_size = 50

source = torch.randint(1, input_vocab_size, (batch_size, seq_len_source))
target = torch.randint(1, target_vocab_size, (batch_size, seq_len_target))


In [7]:
d_model = 512
num_heads = 8
d_ff = 2048
num_layers = 6

model = Transformer(d_model, num_heads, d_ff, num_layers, 
                    input_vocab_size, target_vocab_size, 
                    max_len=MAX_SEQ_LEN, dropout=0.1)

model = model.to(device)
source = source.to(device)
target = target.to(device)

output = model(source, target)

In [8]:
# Expected output shape -> [batch_size, seq len target, target vocab size] i.e. [2, 10, 50]
print(f'output.shape {output.shape}') 

output.shape torch.Size([2, 10, 50])


# Tradutor de Ingles - Portugues

In [6]:
PATH = '/workspaces/transformer/pares_ingles_portugues-2026-05-11.tsv'

In [7]:
!free -h


               total        used        free      shared  buff/cache   available
Mem:            15Gi       4.2Gi       4.3Gi        64Mi       7.5Gi        11Gi
Swap:             0B          0B          0B


In [8]:
with open(PATH, 'r', encoding='utf-8') as f:
    lines = f.readlines()

eng_por_pairs_all = [ line.strip().split('\t') for line in lines if '\t' in line]
eng_por_pairs = [[pair[1], pair[3]] for pair in eng_por_pairs_all]

In [9]:
eng_por_pairs[:4]

[["Let's try something.", 'Vamos tentar alguma coisa!'],
 ["Let's try something.", 'Vamos tentar algo!'],
 ['I have to go to sleep.', 'Preciso ir dormir.'],
 ['I have to go to sleep.', 'Tenho que ir dormir.']]

In [10]:
eng_sentences = [pair[0] for pair in eng_por_pairs]
por_sentences = [pair[1] for pair in eng_por_pairs]

In [11]:
print(eng_sentences[:2])

["Let's try something.", "Let's try something."]


In [12]:
def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[áâãà]+", "a", sentence)
    sentence = re.sub(r"[éêè]+", "e", sentence)
    sentence = re.sub(r"[íîì]+", "i", sentence)
    sentence = re.sub(r"[óôõò]+", "o", sentence)
    sentence = re.sub(r"[úûù]+", "u", sentence)
    sentence = re.sub(r"[ç]+", "c", sentence)
    sentence = re.sub(r"[^a-z]+", " ", sentence)
    sentence = sentence.strip()
    sentence = '<sos> ' + sentence + ' <eos>'
    return sentence

In [13]:
s1 = '?Ola como voce esta? 123'
print(s1)
print(preprocess_sentence(s1))

?Ola como voce esta? 123
<sos> ola como voce esta <eos>


In [14]:
eng_sentences = [preprocess_sentence(sentence) for sentence in eng_sentences]
por_sentences = [preprocess_sentence(sentence) for sentence in por_sentences]

In [15]:
eng_sentences[0:10], por_sentences[0:10]

(['<sos> let s try something <eos>',
  '<sos> let s try something <eos>',
  '<sos> i have to go to sleep <eos>',
  '<sos> i have to go to sleep <eos>',
  '<sos> i have to go to sleep <eos>',
  '<sos> i have to go to sleep <eos>',
  '<sos> today is june th and it is muiriel s birthday <eos>',
  '<sos> today is june th and it is muiriel s birthday <eos>',
  '<sos> today is june th and it is muiriel s birthday <eos>',
  '<sos> today is june th and it is muiriel s birthday <eos>'],
 ['<sos> vamos tentar alguma coisa <eos>',
  '<sos> vamos tentar algo <eos>',
  '<sos> preciso ir dormir <eos>',
  '<sos> tenho que ir dormir <eos>',
  '<sos> preciso dormir <eos>',
  '<sos> tenho de dormir <eos>',
  '<sos> hoje e dia de junho aniversario do muiriel <eos>',
  '<sos> hoje e de junho e e aniversario de muiriel <eos>',
  '<sos> hoje e de junho e e o aniversario de muiriel <eos>',
  '<sos> hoje e de junho e e aniversario de muiriel <eos>'])

In [16]:
def build_vocab(sentences):
    words = [word for sentence in sentences for word in sentence.split()]
    word_count = Counter(words)

    sorted_word_counts = sorted(word_count.items(), key=lambda x:x[1], reverse=True)

    word2idx = {word: idx for idx, (word, _) in enumerate(sorted_word_counts, 2)}
    word2idx['<pad>'] = 0
    word2idx['<unk>'] = 1

    idx2word = {idx: word for word, idx in word2idx.items()}

    return word2idx, idx2word

In [17]:
eng_word2idx, eng_idx2word = build_vocab(eng_sentences)
por_word2idx, por_idx2word = build_vocab(por_sentences)
eng_vocab_size = len(eng_word2idx)
por_vocab_size = len(por_word2idx)

In [18]:
print(eng_vocab_size, por_vocab_size)

41714 54598


In [19]:
class EngPorDataset(Dataset):
    def __init__(self, eng_sentences, por_sentences, eng_word2idx, por_word2idx):
        self.eng_sentences = eng_sentences
        self.por_sentences = por_sentences
        self.eng_word2idx  = eng_word2idx
        self.por_word2idx  = por_word2idx

    def __len__(self):
        return len(self.eng_sentences)
    
    def __getitem__(self, idx):
        eng_sentence = self.eng_sentences[idx]
        por_sentence = self.por_sentences[idx]
        # return tokens idxs
        eng_idxs = [self.eng_word2idx.get(word, self.eng_word2idx['<unk>']) for word in eng_sentence.split()]
        por_idxs = [self.por_word2idx.get(word, self.por_word2idx['<unk>']) for word in por_sentence.split()]

        return torch.tensor(eng_idxs), torch.tensor(por_idxs)

In [20]:
# vamos fazer collate nos vai permitir pasar um argumento a nosso dataloader para fazer padding nas funcoes
def collate_fn(batch):
    """
    # Dados de entrada (lote de frases)
    batch = [
        ("hello", "olá"),
        ("how are you", "como vai"),
        ("good morning", "bom dia")
    ]

    # A mágica acontece aqui
    eng_batch, por_batch = zip(*batch)

    print(eng_batch) # Saída: ('hello', 'how are you', 'good morning')
    print(por_batch) # Saída: ('olá', 'como vai', 'bom dia')"""

    eng_batch, por_batch = zip(*batch)
    eng_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in eng_batch]
    por_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in por_batch]

    #se eng_batch = [tensor([1, 2]), tensor([3, 4, 5])] -->tensor([[1, 2, 0], [3, 4, 5]]) (tamanho 2x3, onde 0 é o preenchimento).
    eng_batch = torch.nn.utils.rnn.pad_sequence(eng_batch, batch_first=True, padding_value=0) 
    por_batch = torch.nn.utils.rnn.pad_sequence(por_batch, batch_first=True, padding_value=0) 
    return eng_batch, por_batch

In [21]:
def train(model, dataloader, loss_function, optimiser, epochs):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for i, (eng_batch, por_batch) in enumerate(dataloader):
            eng_batch = eng_batch.to(device)
            por_batch = por_batch.to(device)

            # decoder preprocessing
            target_input = por_batch[:,:-1]
            target_output = por_batch[:,1:].contiguous().view(-1)

            # zero grads
            optimiser.zero_grad()

            # run model
            output = model(eng_batch, target_input)
            output = output.view(-1, output.size(-1))

            #loss
            loss = loss_function(output, target_output)

            #gradient and update parameters
            loss.backward() # calculo de los gradientes de la funcion de perdida con respecto a todo los parametros
            optimiser.step() # actualizacao dos parametros
            total_loss += loss.item()

        avg_loss = total_loss/len(dataloader)
        print(f'Epoch: {epoch}/{epochs}, Loss: {avg_loss:.4f}')

In [22]:
BATCH_SIZE = 64 #16 #32
dataset = EngPorDataset(eng_sentences, por_sentences, eng_word2idx, por_word2idx)
dataloader = DataLoader(dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

In [23]:
model = Transformer(d_model=512, num_heads=8, d_ff=2048, num_layers=6,
                    input_vocab_size=eng_vocab_size, target_vocab_size=por_vocab_size,
                    max_len=MAX_SEQ_LEN, dropout=0.1)

In [24]:
model = model.to(device)
loss_function = nn.CrossEntropyLoss(ignore_index=0)
optimiser = optim.Adam(model.parameters(), lr=0.0001)


In [25]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            15Gi       5.1Gi       3.4Gi        64Mi       7.5Gi        10Gi
Swap:             0B          0B          0B


In [26]:
train(model, dataloader, loss_function, optimiser, epochs=2)

KeyboardInterrupt: 